# Banana Ripeness — Clean Pool → ConvNeXt → KMeans

1. Load: download (kagglehub cache), stage to `WORK_DIR`, pool all original splits into one.
2. Group by photo: same UUID (Roboflow ×3 augmentations) **or** near-identical pixels (dHash Hamming ≤ `HASH_MAX_DIST`).
3. Drop groups whose files carry conflicting labels.
4. Dedupe: keep one file per group → one clean pool at `CLEAN_DIR/<class>/`.
5. Embed with a barebone ConvNeXt (conv stages + global avg pool, no head).
6. KMeans (`k=4`) on the pool; evaluate clusters against the folder labels.

In [14]:
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import kagglehub as kh
import numpy as np
import torch
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix, normalized_mutual_info_score
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny

In [15]:
# Config
DATASET_SLUG = "shahriar26s/banana-ripeness-classification-dataset"
DATASET_DIR_NAME = "Banana Ripeness Classification Dataset"
WORK_DIR = Path("/tmp/bananafp")
CLEAN_DIR = WORK_DIR / "clean"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
UUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")
HASH_SIZE = 16  # dHash -> 16*16 = 256 bits
HASH_MAX_DIST = 10  # Hamming <= this -> same photo
N_CLUSTERS = 4  # overripe / ripe / rotten / unripe
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

## 1. Load

In [16]:
# Download dataset (cached by kagglehub), stage to work dir
cache_path = Path(kh.dataset_download(DATASET_SLUG))
src = cache_path / DATASET_DIR_NAME if (cache_path / DATASET_DIR_NAME).exists() else cache_path
dataset = WORK_DIR / DATASET_DIR_NAME

if not dataset.exists():
    shutil.copytree(src, dataset)
print(f"Dataset at: {dataset}")

Dataset at: /tmp/bananafp/Banana Ripeness Classification Dataset


In [17]:
# Pool all original splits; keep each file's class (folder name)
image_paths = sorted(p for p in dataset.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
label_of = np.array([p.parent.name for p in image_paths])

print(f"{len(image_paths)} images")
print(Counter(label_of))

13478 images
Counter({np.str_('rotten'): 4593, np.str_('ripe'): 4015, np.str_('overripe'): 2691, np.str_('unripe'): 2179})


## 2. Group by photo (UUID + near-identical pixels)

In [18]:
# Union-find over file indices
parent = list(range(len(image_paths)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(i, j):
    parent[find(i)] = find(j)

In [19]:
# Same UUID -> same photo (Roboflow augmentations share the source UUID)
by_uuid = defaultdict(list)
for i, p in enumerate(image_paths):
    m = UUID_RE.search(p.name)
    by_uuid[m.group() if m else p.name.split(".rf.")[0]].append(i)

for ids in by_uuid.values():
    for j in ids[1:]:
        union(ids[0], j)
print(len(image_paths), "->", len(by_uuid), "UUID groups")

13478 -> 5616 UUID groups


In [20]:
# dHash: near-identical pixels -> same photo (catches re-uploads under new UUIDs)
def dhash(p: Path, n: int = HASH_SIZE) -> np.ndarray:
    a = np.asarray(Image.open(p).convert("L").resize((n + 1, n)), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

H = np.array([dhash(p) for p in image_paths], dtype=np.uint8)

near_pairs = []
for i in range(len(H)):
    d = (H[i + 1 :] != H[i]).sum(1)
    for j in np.nonzero(d <= HASH_MAX_DIST)[0]:
        near_pairs.append((i, i + 1 + int(j)))
        union(i, i + 1 + int(j))
print(len(near_pairs), "near-identical pairs")

493 near-identical pairs


In [21]:
groups = defaultdict(list)
for i in range(len(image_paths)):
    groups[find(i)].append(i)
groups = list(groups.values())

print(len(groups), "photo groups")
print("group sizes:", sorted(Counter(len(g) for g in groups).items()))

5202 photo groups
group sizes: [(1, 1492), (2, 13), (3, 3450), (4, 52), (5, 22), (6, 66), (7, 50), (9, 46), (10, 3), (11, 1), (12, 5), (13, 1), (18, 1)]


## 3. Drop label conflicts

In [22]:
conflicts = [g for g in groups if len(set(label_of[g])) > 1]
for g in conflicts:
    print(sorted(set(label_of[g])), [str(image_paths[i].relative_to(dataset)) for i in g])

groups = [g for g in groups if len(set(label_of[g])) == 1]
print(f"dropped {len(conflicts)} conflicting groups -> {len(groups)} left")

[np.str_('ripe'), np.str_('rotten')] ['test/rotten/musa-acuminata-rotten-89d9466c-2653-11ec-88e8-d8c4975e38aa_jpg.rf.cbc17111a4b86e3ed3b5c94e94da90df.jpg', 'train/ripe/musa-acuminata-ripe-9632290c-1d0a-11ec-81b3-d8c4975e38aa_jpg.rf.0d89b4c43b94c5bbb47fc03b4add0eeb.jpg', 'train/ripe/musa-acuminata-ripe-9632290c-1d0a-11ec-81b3-d8c4975e38aa_jpg.rf.8ea2a7962bacc59713b22755cef851b8.jpg', 'train/ripe/musa-acuminata-ripe-9632290c-1d0a-11ec-81b3-d8c4975e38aa_jpg.rf.e100b609c5914b78cc8304df38718d15.jpg', 'train/rotten/musa-acuminata-ripe-9623dc30-1d0a-11ec-bbce-d8c4975e38aa_jpg.rf.3c499cc3cad3c4a83fa4cd234207f5bb.jpg', 'train/rotten/musa-acuminata-ripe-9623dc30-1d0a-11ec-bbce-d8c4975e38aa_jpg.rf.868a825a516028285e3256475d5a63b3.jpg', 'train/rotten/musa-acuminata-ripe-9623dc30-1d0a-11ec-bbce-d8c4975e38aa_jpg.rf.b69161c277a9230238eca9018ec9c2b7.jpg', 'train/rotten/musa-acuminata-ripe-962fc6fc-1d0a-11ec-adc6-d8c4975e38aa_jpg.rf.08040b1bbd6634b207e01f80eb12d35e.jpg', 'train/rotten/musa-acuminata-ri

## 4. Dedupe → one clean pool

In [23]:
# Keep one representative per group (random, seeded), write CLEAN_DIR/<class>/<file>
rng = random.Random(SEED)
keep = [rng.choice(g) for g in groups]

if CLEAN_DIR.exists():
    shutil.rmtree(CLEAN_DIR)
for i in keep:
    dst = CLEAN_DIR / label_of[i] / image_paths[i].name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_paths[i], dst)

clean_paths = sorted(p for p in CLEAN_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
y = np.array([p.parent.name for p in clean_paths])
assert len(clean_paths) == len(keep), "filename collision in clean pool"
print(len(image_paths), "->", len(clean_paths), "kept at", CLEAN_DIR)
print(Counter(y))

13478 -> 5199 kept at /tmp/bananafp/clean
Counter({np.str_('rotten'): 1676, np.str_('ripe'): 1571, np.str_('overripe'): 1098, np.str_('unripe'): 854})


## 5. Barebone ConvNeXt embeddings

In [24]:
# Conv stages + global avg pool only: drop the whole classifier head
# (LayerNorm2d -> Flatten -> Linear) -> raw (768,) pooled features
weights = ConvNeXt_Tiny_Weights.DEFAULT
preprocess = weights.transforms()
_convnext = convnext_tiny(weights=weights)
backbone = torch.nn.Sequential(_convnext.features, _convnext.avgpool, torch.nn.Flatten(1))
backbone = backbone.eval().to(DEVICE)

@torch.no_grad()
def embed_paths(paths, batch_size: int = 32) -> np.ndarray:  # -> (N, 768)
    vecs = []
    for i in range(0, len(paths), batch_size):
        batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths[i : i + batch_size]])
        vecs.append(backbone(batch.to(DEVICE)).cpu())
    return torch.cat(vecs).numpy()

X = embed_paths(clean_paths)
print("embeddings:", X.shape)

embeddings: (5199, 768)


## 6. KMeans

In [25]:
# L2-normalise so KMeans works on cosine geometry
Xn = X / np.linalg.norm(X, axis=1, keepdims=True).clip(min=1e-9)
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=SEED).fit(Xn)
clusters = kmeans.labels_

# cluster -> label by majority vote
cluster2label = {}
for c in range(N_CLUSTERS):
    vals, counts = np.unique(y[clusters == c], return_counts=True)
    cluster2label[c] = vals[int(np.argmax(counts))]
print("cluster -> label:", cluster2label)

cluster -> label: {0: np.str_('ripe'), 1: np.str_('rotten'), 2: np.str_('overripe'), 3: np.str_('unripe')}


In [26]:
y_pred = np.array([cluster2label[c] for c in clusters])
classes = sorted(set(y))

print(f"majority-vote acc: {(y_pred == y).mean():.3f} (n={len(y)})")
print(f"NMI: {normalized_mutual_info_score(y, clusters):.3f}")
print(f"ARI: {adjusted_rand_score(y, clusters):.3f}")
print("confusion (rows=true, cols=pred):", classes)
print(confusion_matrix(y, y_pred, labels=classes))

majority-vote acc: 0.621 (n=5199)
NMI: 0.338
ARI: 0.219
confusion (rows=true, cols=pred): [np.str_('overripe'), np.str_('ripe'), np.str_('rotten'), np.str_('unripe')]
[[ 681  414    3    0]
 [   8 1507   55    1]
 [ 299  557  820    0]
 [   7  586   39  222]]
